# U-Net basics

U-Net is the standard architecture for image-to-image tasks - segmentation, denoising,
and (as the backbone inside most image diffusion models, including Stable Diffusion). The
shape that matters: an encoder that downsamples and a decoder that upsamples back, with
**skip connections** carrying fine spatial detail directly from encoder to decoder, bypassing
the bottleneck.

We train a small 3-level U-Net on a synthetic segmentation task - find the circle - generated
procedurally, so there's an unlimited supply of training data and no dataset download.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import mlflow
import mlflow.pytorch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

mlflow.set_tracking_uri("file:///work/mlruns")
mlflow.set_experiment("unet-circle-segmentation")

IMG = 64
LR = 1e-3
BATCH_SIZE = 16
_yy, _xx = np.mgrid[0:IMG, 0:IMG]


def make_batch(batch, rng=np.random):
    """A noisy image with one bright circle, and its binary mask."""
    imgs = np.zeros((batch, 1, IMG, IMG), dtype=np.float32)
    masks = np.zeros((batch, 1, IMG, IMG), dtype=np.float32)
    for b in range(batch):
        cx, cy = rng.randint(16, IMG - 16, size=2)
        r = rng.randint(8, 14)
        circle = ((_xx - cx) ** 2 + (_yy - cy) ** 2) <= r * r
        img = rng.randn(IMG, IMG).astype(np.float32) * 0.15
        img[circle] += 1.0
        imgs[b, 0] = img
        masks[b, 0] = circle.astype(np.float32)
    return torch.tensor(imgs), torch.tensor(masks)

## The model

`enc1/enc2/enc3` downsample (each `ConvBlock` doubles the channel count, each `pool` halves
spatial size). `dec2/dec1` upsample back, and at each level the corresponding encoder output
is concatenated in via `torch.cat` before the decoder conv block - that's the skip connection.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class TinyUNet(nn.Module):
    def __init__(self, base=16):
        super().__init__()
        self.enc1 = ConvBlock(1, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.enc3 = ConvBlock(base * 2, base * 4)
        self.pool = nn.MaxPool2d(2)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)
        self.out = nn.Conv2d(base, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        d2 = self.dec2(torch.cat([self.up2(e3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out(d1)  # logits, apply sigmoid for a probability

In [ ]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(log_dir="/work/runs/unet")

rng = np.random.RandomState(0)
model = TinyUNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR)

X_test, Y_test = make_batch(32, rng=rng)
X_test, Y_test = X_test.to(device), Y_test.to(device)


def held_out_iou():
    with torch.no_grad():
        pred = (torch.sigmoid(model(X_test)) > 0.5).float()
        intersection = (pred * Y_test).sum(dim=(1, 2, 3))
        union = ((pred + Y_test) > 0).float().sum(dim=(1, 2, 3))
        return (intersection / union.clamp(min=1)).mean().item()


if mlflow.active_run():
    mlflow.end_run()
mlflow.start_run(run_name="tiny-unet")
mlflow.log_params({
    "image_size": IMG,
    "batch_size": BATCH_SIZE,
    "learning_rate": LR,
    "epochs": 201,
    "test_examples": len(X_test),
})

for epoch in range(201):
    X, Y = make_batch(BATCH_SIZE, rng=rng)
    X, Y = X.to(device), Y.to(device)
    opt.zero_grad()
    logits = model(X)
    loss = F.binary_cross_entropy_with_logits(logits, Y)
    loss.backward()
    opt.step()
    writer.add_scalar("loss/train", loss.item(), epoch)
    mlflow.log_metric("loss_train", loss.item(), step=epoch)
    if epoch % 20 == 0:
        iou_epoch = held_out_iou()
        writer.add_scalar("iou/held_out", iou_epoch, epoch)
        mlflow.log_metric("iou_held_out", iou_epoch, step=epoch)
    if epoch % 50 == 0:
        print(f"epoch {epoch:4d}  loss {loss.item():.4f}")

writer.close()

## Evaluate with IoU

Intersection-over-Union: the overlap between the predicted and true circle, divided by their
union. 1.0 is a perfect match; a model that predicts nothing or everything scores near 0.

In [ ]:
with torch.no_grad():
    logits = model(X_test)
    pred = (torch.sigmoid(logits) > 0.5).float()
    intersection = (pred * Y_test).sum(dim=(1, 2, 3))
    union = ((pred + Y_test) > 0).float().sum(dim=(1, 2, 3))
    iou = (intersection / union.clamp(min=1)).mean().item()
print(f"mean IoU on held-out batch: {iou:.3f}")

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
i = 0
axes[0].imshow(X_test[i, 0].cpu(), cmap="gray"); axes[0].set_title("input")
axes[1].imshow(Y_test[i, 0].cpu(), cmap="gray"); axes[1].set_title("true mask")
axes[2].imshow(pred[i, 0].cpu(), cmap="gray"); axes[2].set_title("predicted mask")
for ax in axes:
    ax.axis("off")
plt.show()

mlflow.log_metric("mean_iou", iou)
mlflow.log_figure(fig, "predictions/example.png")
mlflow.pytorch.log_model(model, "model")
mlflow.end_run()

## Next steps

- Replace `make_batch` with a real dataset loaded from `/work` (e.g. `torchvision.datasets`
  or your own images + masks).
- Add more circles per image, or vary their brightness/contrast, to make the task harder.
- This same encoder-decoder-with-skip-connections shape is the backbone inside image
  diffusion models - see `diffusion.ipynb` (a 1D/2D toy version) for the other half of that
  picture.